In [1]:
%pip install pinecone

from pinecone import Pinecone, ServerlessSpec
import os

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from typing import List, Dict
import numpy as np
import json

In [12]:
from langchain_community.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    """
    Download and Return the HuggingFace Embeddings Model.
    """
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name
    )
    return embeddings

embedding = download_embeddings()

C:\Users\Sadiya Maheen\AppData\Local\Temp\ipykernel_34420\395642784.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [5]:
df = pd.read_csv('../data/Final_Augmented_Dataset_Diseases_and_Symptoms.csv')
df.columns

Index(['diseases', 'anxiety and nervousness', 'depression',
       'shortness of breath', 'depressive or psychotic symptoms',
       'sharp chest pain', 'dizziness', 'insomnia',
       'abnormal involuntary movements', 'chest tightness',
       ...
       'stuttering or stammering', 'problems with orgasm', 'nose deformity',
       'lump over jaw', 'sore in nose', 'hip weakness', 'back swelling',
       'ankle stiffness or tightness', 'ankle weakness', 'neck weakness'],
      dtype='object', length=378)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 246945 entries, 0 to 246944
Columns: 378 entries, diseases to neck weakness
dtypes: int64(377), object(1)
memory usage: 712.2+ MB


In [7]:
df.head()

,diseases,anxiety and nervousness,depression,shortness of breath,depressive or psychotic symptoms,sharp chest pain,dizziness,insomnia,abnormal involuntary movements,chest tightness,...,stuttering or stammering,problems with orgasm,nose deformity,lump over jaw,sore in nose,hip weakness,back swelling,ankle stiffness or tightness,ankle weakness,neck weakness
0,panic disorder,1,0,1,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,panic disorder,0,0,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,panic disorder,1,1,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,panic disorder,1,0,0,1,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
4,panic disorder,1,1,0,0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0


In [24]:
df.describe()

,anxiety and nervousness,depression,shortness of breath,depressive or psychotic symptoms,sharp chest pain,dizziness,insomnia,abnormal involuntary movements,chest tightness,palpitations,...,stuttering or stammering,problems with orgasm,nose deformity,lump over jaw,sore in nose,hip weakness,back swelling,ankle stiffness or tightness,ankle weakness,neck weakness
count,246945.000000,246945.000000,246945.000000,246945.000000,246945.000000,246945.000000,246945.000000,246945.000000,246945.000000,246945.000000,...,246945.0,246945.0,246945.0,246945.0,246945.000000,246945.0,246945.0,246945.0,246945.000000,246945.0
mean,0.039235,0.042746,0.086440,0.061001,0.097252,0.069943,0.039410,0.040572,0.037871,0.024876,...,0.0,0.0,0.0,0.0,0.001385,0.0,0.0,0.0,0.000073,0.0
std,0.194155,0.202285,0.281014,0.239333,0.296302,0.255051,0.194568,0.197296,0.190884,0.155747,...,0.0,0.0,0.0,0.0,0.037189,0.0,0.0,0.0,0.008537,0.0
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0
75%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,0.0,0.0,0.0,0.0,1.000000,0.0,0.0,0.0,1.000000,0.0


In [8]:
df.isnull().sum()

diseases                            0
anxiety and nervousness             0
depression                          0
shortness of breath                 0
depressive or psychotic symptoms    0
                                   ..
hip weakness                        0
back swelling                       0
ankle stiffness or tightness        0
ankle weakness                      0
neck weakness                       0
Length: 378, dtype: int64

In [9]:
knowledge_base = []

for _, row in df.iterrows():
    condition = row['diseases']
    symptoms = [col for col in df.columns if col != 'diseases' and row[col] == 1]
    description = f"Disease: {condition}. Common Symptoms: {','.join(symptoms)}."
    
    knowledge_base.append({"condition" : condition, "info": description})

In [13]:
print(len(knowledge_base))
print(knowledge_base[0])

246945
{'condition': 'panic disorder', 'info': 'Disease: panic disorder. Common Symptoms: anxiety and nervousness,shortness of breath,depressive or psychotic symptoms,chest tightness,palpitations,irregular heartbeat,breathing fast.'}


In [16]:
from langchain.schema import Document

documents = [
    Document(
        page_content=entry["info"],
        metadata={"condition": entry["condition"]}
    )
    for entry in knowledge_base
]

In [17]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key="pcsk_5xq91D_6a4Xccv4CwUPj8MJeebc4wUfqdAvgKt4WNmZpgY42hDWZFZJEXPRfRQYXt7LZie")

if not pc.has_index("symptom-index"):
    pc.create_index(
        name="symptom-index",
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index("symptom-index")

In [18]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=documents,
    embedding=embedding,
    index_name="symptom-index"
)

In [19]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_existing_index(
    index_name="symptom-index",
    embedding=embedding
)

In [33]:
from langchain_pinecone import PineconeVectorStore
from langchain.schema import Document

def retrieve_conditions(query: str, top_k: int = 5):
    retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": top_k})
    
    docs = retriever.get_relevant_documents(query)
    
    matches_with_scores = docsearch.similarity_search_with_score(query, k=top_k)
    
    results = []
    for doc, score in matches_with_scores:
        results.append({
            "score": round(score, 3),
            "metadata": doc.metadata
        })
    
    return results

In [51]:
matches = retrieve_conditions("fever chills body pain")

seen = set()
for m in matches:
    disease_name = m['metadata']['condition'].title()
    if disease_name not in seen:
        seen.add(disease_name)
        score_percent = round(m['score'] * 100, 1)
        print(f"{score_percent}% {disease_name}")


69.6% Flu
67.5% Sepsis
66.9% Common Cold
